# Speculative Decoding

**难度:** Hard

实现推测解码以加速 LLM 推理。

推测解码使用快速草稿模型提议 token，然后根据概率比率与目标模型验证，接受或重新采样。

**签名:** `speculative_decode(target_probs, draft_probs, draft_tokens) -> list[int]`

**参数:**
- `target_probs` — 目标模型概率 (K, V)
- `draft_probs` — 草稿模型概率 (K, V)
- `draft_tokens` — 提议的 token ID (K,)

**返回:** 接受的 token ID 列表（长度 1 到 K）

**约束:**
- 以概率 `min(1, p_target/p_draft)` 接受
- 拒绝时从归一化的 `max(0, p_target - p_draft)` 重新采样
- 在首次拒绝时停止（返回已接受的 + 重采样 token）

**提示:**
for i in range(K):
  accept_prob = min(1, p_target[i, token] / p_draft[i, token])
  若接受：追加 token，继续
  若拒绝：从归一化的 max(0, p_target[i] - p_draft[i]) 重采样，追加后返回

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ SOLUTION

def speculative_decode(target_probs, draft_probs, draft_tokens):
    K = len(draft_tokens)  # 草稿模型提议的 token 数量
    accepted = []

    for i in range(K):
        t = draft_tokens[i].item()  # 当前要验证的 token

        # ---- Step 1: 计算接受概率 ----
        # 接受概率 = min(1, p_target / p_draft)
        # 直觉：如果草稿模型猜对了（p_target 大、p_draft 也大），ratio ≈ 1，大概率接受
        # 如果草稿猜错了（p_target 小、p_draft 大），ratio 很小，大概率拒绝
        # max(..., 1e-10) 防止除零
        ratio = target_probs[i, t] / max(draft_probs[i, t].item(), 1e-10)

        # ---- Step 2: 采样决定是否接受 ----
        if torch.rand(1).item() < min(1.0, ratio.item()):
            # 接受：直接使用草稿模型提议的 token
            accepted.append(t)
        else:
            # ---- Step 3: 拒绝 → 从修正分布重采样 ----
            # 修正分布 = max(0, p_target - p_draft)
            # 为什么？这是保证最终输出分布 = p_target 的数学推导结果
            # 直觉：移除草稿模型已经"用掉"的概率质量
            adjusted = torch.clamp(target_probs[i] - draft_probs[i], min=0)
            s = adjusted.sum()
            if s > 0:
                adjusted = adjusted / s  # 归一化为合法概率分布
            else:
                # 极端情况：如果修正后全为 0，退化为均匀分布
                adjusted = torch.ones_like(adjusted) / adjusted.shape[0]
            # 从修正分布中采样一个 token
            accepted.append(torch.multinomial(adjusted, 1).item())
            # 首次拒绝后立即停止（不再验证后续 token）
            return accepted

    return accepted

In [ ]:
# Demo
torch.manual_seed(0)
probs = torch.softmax(torch.randn(4, 10), dim=-1)
tokens = torch.tensor([2, 5, 1, 8])
print('Perfect draft:', speculative_decode(probs, probs, tokens))

In [ ]:
from torch_judge import check
check('speculative_decoding')

## 📝 核心思路总结

1. **推测解码的核心思想**：用小模型快速"猜"多个 token，大模型一次性验证，拒绝时修正
2. **接受概率 = min(1, p_target/p_draft)**：草稿模型概率越高越容易被拒绝（防止偏差）
3. **拒绝时的修正分布**：max(0, p_target - p_draft) 确保最终输出分布与纯目标模型一致